In [1]:
import numpy as np
import random as rand
import matplotlib.pyplot as plt
import itertools
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
import os
import json
from ELO import Elo

In [2]:
from tqdm import tqdm

In [3]:
def load_training_data(scenario, split):
    base_path = "../Simulated_Data"
    
    filename = f"scenario_{scenario}_{split}_data.csv"
    filepath = os.path.join(base_path, filename)
    
    return pd.read_csv(filepath)

In [4]:
def construct_training_data(df, initial_id, last_id, num_task):
    X = []
    Y = []

    task_ids    = sorted(df['task'].unique())
    task_to_idx = {task: idx for idx, task in enumerate(task_ids)}

    for student_id in range(initial_id, last_id):
        student_df = df[df['student_id'] == student_id].reset_index(drop=True)
        X_student = [torch.zeros(num_task * 2)] 
        Y_student = []

        for _, row in student_df.iterrows():
            task_id = row['task']
            success = row['success']
            y_student = torch.tensor([int(task_id), float(success)])
            Y_student.append(y_student)

            x_student = torch.zeros(num_task *2)
            if success == 1:
                x_student[task_id] = 1
            else:
                x_student[task_id + num_task] = 1
            X_student.append(x_student)
        
        X_student = torch.stack(X_student)
        Y_student = torch.stack(Y_student)
        X.append(X_student)
        Y.append(Y_student)

    return X, Y


In [5]:
import torch
import random

class DKT(torch.nn.Module):
    def __init__(self, num_exercises, num_neurons = 32, C = 1000., num_epochs = 5000, learning_rate = 1E-2):
        super(DKT, self).__init__()
        self.num_exercises_ = num_exercises
        self.num_neurons_   = num_neurons
        self.C_             = C
        self.num_epochs_    = num_epochs
        self.learning_rate_ = learning_rate
        self.gru_           = torch.nn.GRU(self.num_exercises_ * 2, self.num_neurons_) # this is f
        self.out_           = torch.nn.Linear(self.num_neurons_, self.num_exercises_) # this is g

    def forward(self, X):
        H, _ = self.gru_(X)
        return self.out_(H)

    """
    Scores the accuracy of this model.
    """
    def score(self, X, Y):
        num_attempts  = 0
        num_successes = 0
        for i in range(len(X)):
            Yi = Y[i]
            Logits = self.forward(X[i])
            selected_logits = Logits.gather(
                                  1, Yi[:,0].long().unsqueeze(1)
                              ).squeeze(1)
            probs = torch.sigmoid(selected_logits)
            preds = (probs >= 0.5).float()

            num_successes += (preds == Yi[:,1]).sum().item()
            num_attempts += len(Yi[:,1])
        return num_successes / num_attempts
    
    def waviness_terms(self, Logits):
        probs = torch.sigmoid(Logits)          # (T, M)
        diff  = probs[1:] - probs[:-1]         # (T-1, M)
        M  = Logits.shape[1]
        Tm1 = diff.shape[0]
        w1     = diff.abs().sum() / (M * Tm1)
        w2_sq  = (diff ** 2).sum() / (M * Tm1)   # this is w2^2 as in eq. 7; take sqrt if you want w2 itself
        return w1, w2_sq
    
    @torch.no_grad()
    def waviness(self, X):
        self.eval()
        total_l1 = 0.0
        total_l2 = 0.0
        total_denom = 0.0
        for Xi in X:
            Logits = self.forward(Xi)
            w1, w2_sq = self.waviness_terms(Logits)
            n = (Logits.shape[0] - 1) * Logits.shape[1]
            total_l1 += w1.item() * n
            total_l2 += w2_sq.item() * n
            total_denom += n
        self.train()
        return total_l1 / total_denom, (total_l2 / total_denom) ** 0.5

    """
    Fits this model to the given data matrix.

    Parameters
    ----------
    X: list
        A list of feature matrices, one per student.
    Y: list
        A list of target matrices, one per student.

    """
    def fit(self, X, Y):
        # initialize optimizer
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate_, weight_decay = 1. / self.C_)
        self.train()
        # start training
        for epoch in range(self.num_epochs_):
            # re-set the current gradient to zero
            optimizer.zero_grad()
            # get a random student index
            i  = random.randrange(len(X))
            Xi = X[i]
            Yi = Y[i]
            # compute the Logits
            Logits = self.forward(Xi)
            selected_logits = Logits.gather(
                                  1, Yi[:,0].long().unsqueeze(1)
                              ).squeeze(1) # Choose the index of the executed task, and only take the logit of this task for every time step
            # compute the loss
            loss = torch.nn.functional.binary_cross_entropy_with_logits(selected_logits, Yi[:,1].float())
            # report the loss
            if (epoch+1) % 1000 == 0:
                print('epoch %d: loss %g' % (epoch+1, loss.item()))
            # compute the gradient
            loss.backward()
            # apply the optimizer
            optimizer.step()

        return self

In [6]:
def probability(theta, b):
    p = 1 / (1 + np.exp(-(theta - b)))
    return p

In [7]:
def save_model(model, scenario_id, num_exercises, base_dir=None):
    """
    Save a trained model checkpoint for a given scenario.

    Args:
        model       : the trained PyTorch model
        scenario_id : scenario identifier (e.g. 1–6, or a string name)
        learning_rate, hidden_size, batch_size : hyperparameters to store
        base_dir    : optional override for the Trained_Models folder path
    """
    if base_dir is None:
        base_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'Trained_Models'))

    os.makedirs(base_dir, exist_ok=True)

    model_path = os.path.join(base_dir, f'DKT_model_scenario_{scenario_id}.pth')
    torch.save({
        'scenario'     : scenario_id,
        'model_state'  : model.state_dict(),
        'num_exercises': num_exercises,
        'num_neurons'  : model.num_neurons_,
        'C'            : model.C_,
        'num_epochs'   : model.num_epochs_,
        'learning_rate': model.learning_rate_,
    }, model_path)

    print(f"[Scenario {scenario_id}] Model saved → {model_path}")
    return model_path

# Start Training

In [8]:
with open('../elo_variable.json', 'r') as Elo_Data:
    elo_data = json.load(Elo_Data)

print(elo_data)

{'globals': {'students': 500, 'init_skill_level': [0.0, 0.0], 'k_success': 1, 'k_fail': 0.5}, 'scenarios': [{'id': 1, 'num_tasks': 2, 'num_skills': 2, 'q_matrix': [[1, 0], [0, 1]], 'difficulty_level': [1, 1], 'depends': [-1, -1]}, {'id': 2, 'num_tasks': 2, 'num_skills': 2, 'q_matrix': [[1, 1], [1, 1]], 'difficulty_level': [0, 1], 'depends': [-1, -1]}, {'id': 3, 'num_tasks': 4, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 0], [0, 1, 0, 1]], 'difficulty_level': [0, 0, 1, 1], 'depends': [-1, -1, -1, -1]}, {'id': 4, 'num_tasks': 4, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 1], [0, 1, 1, 1]], 'difficulty_level': [1, 1, 0, 1], 'depends': [-1, -1, -1, -1]}, {'id': 5, 'num_tasks': 6, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 1, 0, 1], [0, 1, 1, 0, 1, 1]], 'difficulty_level': [0, 0, 0, 1, 1, 1], 'depends': [-1, -1, -1, -1, -1, -1]}, {'id': 6, 'num_tasks': 6, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1]], 'difficulty_level': [0, 0, 1, 1, 2, 2], 'depends': [-1, -1, -1, -1, -1, -1]}

In [9]:
all_data = {}

level_skill   = [] 
mastery_level = 1.5
i = 0

global_values = elo_data["globals"]
scenarios = elo_data["scenarios"]

students          = global_values["students"]
init_skill_level  = np.array(global_values["init_skill_level"])
k_success         = global_values["k_success"]
k_fail            = global_values["k_fail"]

In [48]:
for scenario in scenarios:
    scenario_id    = scenario["id"]
    num_tasks      = scenario["num_tasks"]
    difficulty_level = np.array(scenario["difficulty_level"])
    q_matrix       = np.array(scenario["q_matrix"])
    depends = np.array(scenario["depends"])
    results = [] 
    results_mod = []

    df = load_training_data(scenario_id, "train")
    X_train, Y_train   = construct_training_data(df, 0, 400, num_tasks)
    dkt = DKT(num_tasks)
    dkt.fit(X_train, Y_train)
    accuracy = dkt.score(X_train, Y_train)
    print(f"Scenario {scenario_id}: Accuracy {accuracy} \n")

    val_df = load_training_data(scenario_id, "vali")
    X_val, Y_val   = construct_training_data(val_df, 450, 500, num_tasks)
    val_accuracy = dkt.score(X_val, Y_val)
    wav_l1, wav_l2 = dkt.waviness(X_val)
    print(f"Scenario {scenario_id}: val accuracy {val_accuracy} wav l1 {wav_l1} wav l2 {wav_l2}\n")

    save_model(dkt, scenario_id, num_tasks)
    

epoch 1000: loss 0.53697
epoch 2000: loss 0.607613
epoch 3000: loss 0.409698
epoch 4000: loss 0.711591
epoch 5000: loss 0.889618
Scenario 1: Accuracy 0.7156644394951744 

Scenario 1: val accuracy 0.7137476459510358 wav l1 0.06761916745769775 wav l2 0.0953042973107447

[Scenario 1] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/DKT_model_scenario_1.pth
epoch 1000: loss 0.288606
epoch 2000: loss 0.234192
epoch 3000: loss 0.261916
epoch 4000: loss 0.237706
epoch 5000: loss 0.31484
Scenario 2: Accuracy 0.8556895252449134 

Scenario 2: val accuracy 0.847953216374269 wav l1 0.4359527103036468 wav l2 0.4913530398425368

[Scenario 2] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/DKT_model_scenario_2.pth
epoch 1000: loss 0.520351
epoch 2000: loss 0.498235
epoch 3000: loss 0.668623
epoch 4000: loss 0.609032
epoch 5000: loss 0.563254
Scenario 3: Accuracy 0.7638640429338104 

Scenario 3: val accuracy 0.7683923705722071 wav l1 0.041433777962113436 wav l2 0.068620268

In [11]:
for scenario in scenarios:
    scenario_id    = scenario["id"]
    num_tasks      = scenario["num_tasks"]
    difficulty_level = np.array(scenario["difficulty_level"])
    q_matrix       = np.array(scenario["q_matrix"])
    depends = np.array(scenario["depends"])
    results = [] 
    results_mod = []

    df = load_training_data(scenario_id, "train")
    X_train, Y_train   = construct_training_data(df, 0, 400, num_tasks)
    dkt = DKT(num_tasks)
    dkt.fit(X_train, Y_train)
    accuracy = dkt.score(X_train, Y_train)
    print(f"Scenario {scenario_id}: Accuracy {accuracy} \n")

    val_df = load_training_data(scenario_id, "vali")
    X_val, Y_val   = construct_training_data(val_df, 450, 500, num_tasks)
    val_accuracy = dkt.score(X_val, Y_val)
    wav_l1, wav_l2 = dkt.waviness(X_val)
    print(f"Scenario {scenario_id}: val accuracy {val_accuracy} wav l1 {wav_l1} wav l2 {wav_l2}\n")

    save_model(dkt, scenario_id, num_tasks)
    

epoch 1000: loss 0.744399
epoch 2000: loss 0.656947
epoch 3000: loss 0.554537
epoch 4000: loss 0.823542
epoch 5000: loss 0.660493
Scenario 1: Accuracy 0.6396843291995491 

Scenario 1: val accuracy 0.6277777777777778 wav l1 0.0425249208247772 wav l2 0.0546683253484894

[Scenario 1] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/DKT_model_scenario_1.pth
epoch 1000: loss 0.362788
epoch 2000: loss 0.200032
epoch 3000: loss 0.316411
epoch 4000: loss 0.197951
epoch 5000: loss 0.226532
Scenario 2: Accuracy 0.8479020979020979 

Scenario 2: val accuracy 0.8620071684587813 wav l1 0.3056622369208216 wav l2 0.3500347142942243

[Scenario 2] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/DKT_model_scenario_2.pth
epoch 1000: loss 0.743438
epoch 2000: loss 0.562349
epoch 3000: loss 0.571998
epoch 4000: loss 0.438965
epoch 5000: loss 0.851118
Scenario 3: Accuracy 0.6637651821862348 

Scenario 3: val accuracy 0.6521035598705501 wav l1 0.044655865950443595 wav l2 0.0595934

Interpretation:
- DKT does pick the task with the highest learning gain, but sometimes it forgets that there is another skill to be trained that is just as important (see scenario 1 and 3). We need to limit the growth of a certain skill if it is already mastered
- DKT tends to favor the easier task (see scenario 2 and 3) if there are several tasks training the same skill
- DKT works better with more task variations?
- Sometimes we#re lucky sometimes we're not